In [55]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import numpy as np
from sklearn.model_selection import cross_validate,GridSearchCV, RandomizedSearchCV
from scipy.stats import randint, loguniform
from datetime import datetime
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
import joblib
import sklearn.svm as svm

In [35]:
train_set = pd.read_csv("../data/processed/train_set.csv")
test_set = pd.read_csv("../data/processed/test_set.csv")

In [36]:
# Define Features and Target Variable

TARGET = 'price'
X_train = train_set.drop(columns=[TARGET])
y_train = train_set[TARGET]

X_test = test_set.drop(columns = [TARGET])
y_test = test_set[TARGET]

In [37]:
# Feature Preprocessing Pipeline
cat_cols = X_train.select_dtypes(include = ['object', 'category','str']).columns.tolist()
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

numeric_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', StandardScaler())
])

categorical_pipe = Pipeline (steps=[
    ('imputer', SimpleImputer (strategy= 'constant', fill_value= 'missing') ),
    ('ohe', OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency= 0.01
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipe, num_cols),
        ('cat', categorical_pipe, cat_cols),
    ],
    remainder = 'drop'
)

In [56]:
# Cross-Validation Evaluation and Experiment Logging

def regression_cv_metrics(model, X, y, cv=5):
  scores = cross_validate(
      model, X, y,
      scoring= {'rmse':'neg_root_mean_squared_error',
                'mae':'neg_mean_absolute_error',
                'mape':'neg_mean_absolute_percentage_error'
                },
      cv =cv,
      n_jobs =-1
  )

  rmse_scores = -scores['test_rmse']
  mae_scores = -scores['test_mae']
  mape_scores = -scores['test_mape']

  return {'rmse_mean': rmse_scores.mean(),
      'rmse_std':rmse_scores.std(),
      'mae_mean':mae_scores.mean(),
      'mae_std': mae_scores.std(),
      'mape_mean': mape_scores.mean(),
      'mape_std':mape_scores.std(),
}

results = []

def log_result(variant, model_name, rmse_mean, rmse_std, mae_mean,mae_std, mape_mean, mape_std, notes=''):
  results.append({
      'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M'),
      'variant': variant,
      'model': model_name,
      'rmse_mean': rmse_mean,
      'rmse_std':rmse_std,
      'mae_mean':mae_mean,
      'mae_std': mae_std,
      'mape_mean': mape_mean,
      'mape_std':mape_std,
      'notes': notes
  })

In [39]:
lin_reg = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', LinearRegression())
])

metrics = regression_cv_metrics(lin_reg, X_train, y_train, cv=5)

log_result(
    'BaseLine',
    'LinearRegression',
    metrics['rmse_mean'],
    metrics['rmse_std'],
    metrics['mae_mean'],
    metrics['mae_std'],
    metrics['mape_mean'],
    metrics['mape_std']
)

pd.DataFrame(results).sort_values('rmse_mean')

,timestamp,variant,model,rmse_mean,rmse_std,mae_mean,mae_std,mape_mean,mape_std,notes
0,2026-03-08 13:07,BaseLine,LinearRegression,7147.805028,25.337156,5234.397852,6.854459,0.515079,0.005371,


In [40]:
# Baseline 2. DecisionTreeRegressor

tree_reg = Pipeline(steps = [
    ('preprocess', preprocessor),
    ('model', DecisionTreeRegressor(random_state=42))
])
metrics = regression_cv_metrics(tree_reg, X_train, y_train, cv=5)

log_result(
    'BaseLine 2',
    'DecisionTreeRegressor',
    metrics['rmse_mean'],
    metrics['rmse_std'],
    metrics['mae_mean'],
    metrics['mae_std'],
    metrics['mape_mean'],
    metrics['mape_std']
)

pd.DataFrame(results).sort_values('rmse_mean')

,timestamp,variant,model,rmse_mean,rmse_std,mae_mean,mae_std,mape_mean,mape_std,notes
1,2026-03-08 13:08,BaseLine 2,DecisionTreeRegressor,5276.646764,28.057011,2319.524621,22.414052,0.223143,0.003674,
0,2026-03-08 13:07,BaseLine,LinearRegression,7147.805028,25.337156,5234.397852,6.854459,0.515079,0.005371,


In [41]:
rf_reg = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', RandomForestRegressor(
        random_state = 42,
        n_estimators=50,
        max_depth=20,
        min_samples_leaf = 5,
        n_jobs=-1
    ))
])

metrics = regression_cv_metrics(rf_reg, X_train, y_train, cv=5)

log_result(
    'BaseLine 3',
    'RandomForestRegressor',
    metrics['rmse_mean'],
    metrics['rmse_std'],
    metrics['mae_mean'],
    metrics['mae_std'],
    metrics['mape_mean'],
    metrics['mape_std']
)

pd.DataFrame(results).sort_values('rmse_mean')

,timestamp,variant,model,rmse_mean,rmse_std,mae_mean,mae_std,mape_mean,mape_std,notes
2,2026-03-08 13:19,BaseLine 3,RandomForestRegressor,4896.218549,36.943654,3035.891829,12.137614,0.280480,0.003797,
1,2026-03-08 13:08,BaseLine 2,DecisionTreeRegressor,5276.646764,28.057011,2319.524621,22.414052,0.223143,0.003674,
0,2026-03-08 13:07,BaseLine,LinearRegression,7147.805028,25.337156,5234.397852,6.854459,0.515079,0.005371,


Random Forest significantly outperformed the linear baseline, reducing RMSE from ~$7148 to ~$4896 (≈31% improvement). Decision Tree achieved lower MAE but exhibited larger variance in predictions, leading to higher RMSE. Random Forest provided the best balance between accuracy and robustness across cross-validation folds.

In [42]:
rf_reg = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', RandomForestRegressor(
        random_state = 42,
        n_estimators=100,
        max_depth=30,
        min_samples_leaf = 2,
        n_jobs=-1
    ))
])

metrics = regression_cv_metrics(rf_reg, X_train, y_train, cv=5)

log_result(
    'BaseLine 4',
    'RandomForestRegressor try to go deeper',
    metrics['rmse_mean'],
    metrics['rmse_std'],
    metrics['mae_mean'],
    metrics['mae_std'],
    metrics['mape_mean'],
    metrics['mape_std']
)

pd.DataFrame(results).sort_values('rmse_mean')

,timestamp,variant,model,rmse_mean,rmse_std,mae_mean,mae_std,mape_mean,mape_std,notes
3,2026-03-08 14:17,BaseLine 4,RandomForestRegressor try to go deeper,4191.447786,38.796376,2316.676717,11.934988,0.224306,0.003345,
2,2026-03-08 13:19,BaseLine 3,RandomForestRegressor,4896.218549,36.943654,3035.891829,12.137614,0.280480,0.003797,
1,2026-03-08 13:08,BaseLine 2,DecisionTreeRegressor,5276.646764,28.057011,2319.524621,22.414052,0.223143,0.003674,
0,2026-03-08 13:07,BaseLine,LinearRegression,7147.805028,25.337156,5234.397852,6.854459,0.515079,0.005371,


In [43]:
rf_reg = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', RandomForestRegressor(
        random_state = 42,
        n_estimators=100,
        max_depth=25,
        min_samples_leaf = 2,
        n_jobs=-1
    ))
])

metrics = regression_cv_metrics(rf_reg, X_train, y_train, cv=5)

log_result(
    'BaseLine 5',
    'RandomForestRegressor try to go deeper',
    metrics['rmse_mean'],
    metrics['rmse_std'],
    metrics['mae_mean'],
    metrics['mae_std'],
    metrics['mape_mean'],
    metrics['mape_std']
)

pd.DataFrame(results).sort_values('rmse_mean')

,timestamp,variant,model,rmse_mean,rmse_std,mae_mean,mae_std,mape_mean,mape_std,notes
3,2026-03-08 14:17,BaseLine 4,RandomForestRegressor try to go deeper,4191.447786,38.796376,2316.676717,11.934988,0.224306,0.003345,
4,2026-03-09 10:24,BaseLine 5,RandomForestRegressor try to go deeper,4374.922383,38.216528,2526.513907,10.926390,0.240710,0.003320,
2,2026-03-08 13:19,BaseLine 3,RandomForestRegressor,4896.218549,36.943654,3035.891829,12.137614,0.280480,0.003797,
1,2026-03-08 13:08,BaseLine 2,DecisionTreeRegressor,5276.646764,28.057011,2319.524621,22.414052,0.223143,0.003674,
0,2026-03-08 13:07,BaseLine,LinearRegression,7147.805028,25.337156,5234.397852,6.854459,0.515079,0.005371,


In [58]:
# Support Vector Regression

X_train_svr = X_train.iloc[:50000].copy()
y_train_svr = y_train.iloc[:50000].copy()

svr_pipe = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', svm.SVR())
])

param_distributions = [{
   'model__kernel' : ['linear'],
    'model__C': loguniform(1, 1000)
},
    {
        'model__kernel': ['rbf'],
        'model__C': loguniform(1, 1000),
        'model__gamma':loguniform(1e-4, 1e-1)
    }
]

svr_search = RandomizedSearchCV(
    svr_pipe,
    param_distributions,
    n_iter=10,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=42
)

svr_search.fit(X_train_svr, y_train_svr)
print("Best params:", svr_search.best_params_)
print("Best RMSE:", -svr_search.best_score_)

svr_metrics = regression_cv_metrics(
    svr_search.best_estimator_,
    X_train_svr,
    y_train_svr,
    cv=3
)

log_result(
    'Baseline 6 Exercise 1 - SVR',
    'SVR',
    svr_metrics['rmse_mean'],
    svr_metrics['rmse_std'],
    svr_metrics['mae_mean'],
    svr_metrics['mae_std'],
    svr_metrics['mape_mean'],
    svr_metrics['mape_std'],
    notes=str(svr_search.best_params_)
)

pd.DataFrame(results).sort_values('rmse_mean')


KeyboardInterrupt



In [ ]:
# Randomized Search Over Preprocessing and Model Parameters
rf_reg = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', RandomForestRegressor(random_state=42, n_jobs=-1))
])

param_distributions = {
    'preprocess__num__imputer__strategy': ['median', 'most_frequent'],
    'preprocess__num__imputer__add_indicator': [True, False],
    'preprocess__cat__ohe__min_frequency': [0.005, 0.01, 0.02],
    'model__n_estimators': randint(50, 150),
    'model__max_depth': randint(15, 35),
    'model__min_samples_leaf': randint(1, 6)
}

rand_search_prep = RandomizedSearchCV(
    rf_reg,
    param_distributions=param_distributions,
    n_iter=10,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=42
)

rand_search_prep.fit(X_train, y_train)

print("Best params:", rand_search_prep.best_params_)
print("Best RMSE:", -rand_search_prep.best_score_)

prep_metrics = regression_cv_metrics(
    rand_search_prep.best_estimator_,
    X_train,
    y_train,
    cv=5
)

log_result(
    'baseline 7 Prep + RF RandomSearch +++',
    'RandomForestRegressor',
    prep_metrics['rmse_mean'],
    prep_metrics['rmse_std'],
    prep_metrics['mae_mean'],
    prep_metrics['mae_std'],
    prep_metrics['mape_mean'],
    prep_metrics['mape_std'],
    notes=str(rand_search_prep.best_params_)
)

pd.DataFrame(results).sort_values('rmse_mean')

In [53]:
# Grid Search Hyperparameter Tuning
param_grid = {
    'model__n_estimators': [50, 100],
    'model__max_depth': [20, 30],
    'model__min_samples_leaf': [2, 5]
}
grid_search = GridSearchCV(
    rf_reg,
    param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print('Best params:', grid_search.best_params_)
print('Best RMSE:', -grid_search.best_score_)

grid_metrics = regression_cv_metrics(
    grid_search.best_estimator_,
    X_train,
    y_train,
    cv=5
)

log_result(
    'baseline 8 Prep + RF GridSearch',
    'RandomForestRegressor',
    prep_metrics['rmse_mean'],
    prep_metrics['rmse_std'],
    prep_metrics['mae_mean'],
    prep_metrics['mae_std'],
    prep_metrics['mape_mean'],
    prep_metrics['mape_std'],
    notes=str(grid_search.best_params_)
)

pd.DataFrame(results).sort_values('rmse_mean')


KeyboardInterrupt



In [52]:
# Randomized Hyperparameter Search

param_distributions = {
    'model__n_estimators': randint(50, 150),
    'model__max_depth': randint(15, 35),
    'model__min_samples_leaf': randint(1, 6)
}

rand_search = RandomizedSearchCV(
    rf_reg,
    param_distributions,
    n_iter=10,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=42
)

rand_search.fit(X_train, y_train)

print('Best params:', rand_search.best_params_)
print('Best RMSE:', -rand_search.best_score_)

rand_metrics = regression_cv_metrics(
    rand_search.best_estimator_,
    X_train,
    y_train,
    cv=5
)

log_result(
    'baseline 9 RF Radom Search',
    'RandomForestRegressor',
    prep_metrics['rmse_mean'],
    prep_metrics['rmse_std'],
    prep_metrics['mae_mean'],
    prep_metrics['mae_std'],
    prep_metrics['mape_mean'],
    prep_metrics['mape_std'],
    notes=str(grid_search.best_params_)
)

pd.DataFrame(results).sort_values('rmse_mean')

KeyboardInterrupt: 

In [61]:
print("Grid RMSE:", -grid_search.best_score_)
print("Random RMSE:", -rand_search.best_score_)

AttributeError: 'GridSearchCV' object has no attribute 'best_score_'

In [ ]:
if -grid_search.best_score_ < -rand_search.best_score_:
    final_model = grid_search.best_estimator_
    best_search_name = "GridSearchCV"
else:
    final_model = rand_search.best_estimator_
    best_search_name = "RandomizedSearchCV"

print("Selected search:", best_search_name)

In [60]:
# Best Model and Feature Importance

feature_importances = final_model['model'].feature_importances_
feature_importances.round(3)

NameError: name 'final_model' is not defined

In [ ]:
 # Ranking Features by Importance
feature_name = final_model['preprocess'].get_feature_names_out()
sorted(zip(feature_importances, feature_name), reverse=True)

In [ ]:
# Feature Selection Using Model Importances
selector = SelectFromModel(final_model['model'], threshold='median')
X_reduces = selector.fit_transform(final_model['preprocess'].transform(X_train),y_train)

In [ ]:
# final predictions
final_predictions = final_model.predict(X_test)

# calculate metrics

final_rmse = root_mean_squared_error(y_test, final_predictions)
final_mae = mean_absolute_error(y_test, final_predictions)
final_mape = mean_absolute_percentage_error(y_test, final_predictions)

print('Final RMSE:', final_rmse)
print('Final MAE:', final_mae)
print('Final MAPE:', final_mape)
print('Final MAPE:', final_mape*100)

In [ ]:
# Saving the Final Model
joblib.dump(final_model, '../models/final_model.joblib')